In [6]:

import mat2qubit as m2q
from openfermion import QubitOperator
import numpy as np
import itertools
import qutip as qt

In [7]:

# Initialize a symbolic operator

omega_c = 1.0
omega_q = 1.0
g = 0.1
J = 0.05
M = 3   # number of sites
N = 4   # number of bosonic levels per site
d = 8

jch_model = m2q.qSymbOp("")

for i in range(M):
    # next site for hopping (open boundary: no hopping from last to first)
    j = i + 1

    # cavity (bosonic) energy
    jch_model += m2q.qSymbOp(f" omega_c [n_{i}]")

    # qubit energy (Pauli Z or sigma+ sigma-)
    jch_model += m2q.qSymbOp(f" omega_q [Sz_{i}]")  

    # light–matter coupling g (Jaynes–Cummings term)
    jch_model += m2q.qSymbOp(f" g [ad_{i} Sm_{i}] ++ g [a_{i} Sp_{i}]")

    # hopping between cavities (only if not the last site)
    if j < M:
        jch_model += m2q.qSymbOp(f" -J [a_{i} ad_{j}] ++ -J [ad_{i} a_{j}]")

# Substitute numerical values
vals = {'omega_c': omega_c, 'omega_q': omega_q, 'g': g, 'J': J}
jch_model.scalar_subs(vals)
print("\nJaynes–Cummings–Hubbard model:")
print(jch_model)



print('''
The symbolic operator for a 3-particle JCH model
(defined before any choice of d [# levels] or encoding)''')
# print(jch_model)

a = (qt.destroy(N)).full()

# Create the raising (creation) operator for the cavity mode
adag = (qt.create(N)).full()


Sm = np.kron(np.array([[0,0],[1,0]]), np.eye(N))
Sp = np.kron(np.array([[0,1],[0,0]]), np.eye(N))
Sz = np.kron(np.array([[1,0],[0,-1]]), np.eye(N))
n = np.kron(np.eye(2), np.diag(np.arange(N)))
ad = np.kron(np.eye(2), adag)
a = np.kron(np.eye(2), a)
print(Sm.shape)
print(Sp.shape)
print(Sz.shape)
print(n.shape)
print(ad.shape)
print(a.shape)
inpOps = {}
inpOps['Sm'] = Sm
inpOps['Sp'] = Sp
inpOps['Sz'] = Sz
inpOps['n'] = n
inpOps['ad'] = ad
inpOps['a'] = a
# for i in range(M):
#     inpOps[f"Sm_{i}"] = Sm
#     inpOps[f"Sp_{i}"] = Sp

print("\nConvert to mat2qubit compositeOperator")
ssid_order = [str(i) for i in range(M)] # ['0','1','2']
dlev_obj = m2q.symbop_to_dlevcompositeop(jch_model, ssid_order,d,'stdbinary',inpOpChars=inpOps )
# print(dlev_obj)

print("\nConvert to QubitOperator")
qub_op = dlev_obj.opToPauli()
print(qub_op)


Jaynes–Cummings–Hubbard model:
((1+0j)) [n_0]
++ ((1+0j)) [Sz_0]
++ ((0.1+0j)) [ad_0 Sm_0]
++ ((0.1+0j)) [a_0 Sp_0]
++ ((-0.05+0j)) [a_0 ad_1]
++ ((-0.05+0j)) [ad_0 a_1]
++ ((1+0j)) [n_1]
++ ((1+0j)) [Sz_1]
++ ((0.1+0j)) [ad_1 Sm_1]
++ ((0.1+0j)) [a_1 Sp_1]
++ ((-0.05+0j)) [a_1 ad_2]
++ ((-0.05+0j)) [ad_1 a_2]
++ ((1+0j)) [n_2]
++ ((1+0j)) [Sz_2]
++ ((0.1+0j)) [ad_2 Sm_2]
++ ((0.1+0j)) [a_2 Sp_2]

The symbolic operator for a 3-particle JCH model
(defined before any choice of d [# levels] or encoding)
(8, 8)
(8, 8)
(8, 8)
(8, 8)
(8, 8)
(8, 8)

Convert to mat2qubit compositeOperator

Convert to QubitOperator
4.5 [] +
(0.03535533905932738+0j) [X0 X1 X2] +
(-0.02414814565722671+0j) [X0 X1 X3] +
(-0.012500000000000004+0j) [X0 X1 X3 X4] +
(0.0064704761275630185+0j) [X0 X1 X3 Z4] +
(-0.012500000000000004+0j) [X0 X1 Y3 Y4] +
(-0.03535533905932738+0j) [X0 Y1 Y2] +
(-0.012500000000000004+0j) [X0 Y1 X3 Y4] +
(-0.02414814565722671+0j) [X0 Y1 Y3] +
(0.012500000000000004+0j) [X0 Y1 Y3 X4] +
(0.0064

In [8]:
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import CountOps
from qiskit import transpile
from qiskit.circuit.library import PauliEvolutionGate
from qiskit import QuantumCircuit

In [9]:
# Convert mat2qubit -> Qiskit
paulis = []
coeffs = []
for term, coeff in qub_op.terms.items():
    # term looks like (('X', 0), ('Z', 1)) etc.
    max_qubit = max([q for q, _ in term]) if term else 0
    num_qubits = 9

    # Initialize all to 'I'
    label = ['I'] * num_qubits
    for q, op in term:
        label[q] = op
    paulis.append(''.join(label[::-1]))  # Qiskit uses reversed order
    coeffs.append(coeff.real)

H_op = SparsePauliOp(paulis, coeffs)

# Create circuit for time evolution
theta = 0.1
evol_gate = PauliEvolutionGate(H_op, time=theta)
qc = QuantumCircuit(H_op.num_qubits)
qc.append(evol_gate, range(H_op.num_qubits))

# Fully decompose and count
qc_decomp = qc.decompose(reps=100)
print(qc_decomp.count_ops())
# qc_decomp.draw('mpl')

OrderedDict([('u', 1065), ('cx', 404)])


In [10]:
from qiskit import transpile

# Transpile for gate optimization
qc_opt = transpile(
    qc,
    optimization_level=3,   # 0-3, 3 = heaviest optimization
    basis_gates=['u3','cx'],  # or your hardware basis
    routing_method='sabre'
)

print(qc_opt.count_ops())

OrderedDict([('cx', 393), ('u3', 265)])


In [11]:
from qiskit_optimization.applications import Knapsack
from qiskit_optimization.converters import QuadraticProgramToQubo

In [12]:
prob = Knapsack(values=[3, 4, 5, 6, 7], weights=[2, 3, 4, 5, 6], max_weight=10)
qp = prob.to_quadratic_program()
print(qp.prettyprint())



Problem name: Knapsack

Maximize
  3*x_0 + 4*x_1 + 5*x_2 + 6*x_3 + 7*x_4

Subject to
  Linear constraints (1)
    2*x_0 + 3*x_1 + 4*x_2 + 5*x_3 + 6*x_4 <= 10  'c0'

  Binary variables (5)
    x_0 x_1 x_2 x_3 x_4



In [13]:
conv = QuadraticProgramToQubo()
qubo = conv.convert(qp)
op, offset = qubo.to_ising()
print(f"num qubits: {op.num_qubits}, offset: {offset}\n")
print(op)

num qubits: 9, offset: 1417.5

SparsePauliOp(['IIIIIIIIZ', 'IIIIIIIZI', 'IIIIIIZII', 'IIIIIZIII', 'IIIIZIIII', 'IIIZIIIII', 'IIZIIIIII', 'IZIIIIIII', 'ZIIIIIIII', 'IIIIIIIZZ', 'IIIIIIZIZ', 'IIIIIZIIZ', 'IIIIZIIIZ', 'IIIZIIIIZ', 'IIZIIIIIZ', 'IZIIIIIIZ', 'ZIIIIIIIZ', 'IIIIIIZZI', 'IIIIIZIZI', 'IIIIZIIZI', 'IIIZIIIZI', 'IIZIIIIZI', 'IZIIIIIZI', 'ZIIIIIIZI', 'IIIIIZZII', 'IIIIZIZII', 'IIIZIIZII', 'IIZIIIZII', 'IZIIIIZII', 'ZIIIIIZII', 'IIIIZZIII', 'IIIZIZIII', 'IIZIIZIII', 'IZIIIZIII', 'ZIIIIZIII', 'IIIZZIIII', 'IIZIZIIII', 'IZIIZIIII', 'ZIIIZIIII', 'IIZZIIIII', 'IZIZIIIII', 'ZIIZIIIII', 'IZZIIIIII', 'ZIZIIIIII', 'ZZIIIIIII'],
              coeffs=[-258.5+0.j, -388. +0.j, -517.5+0.j, -647. +0.j, -776.5+0.j, -130. +0.j,
 -260. +0.j, -520. +0.j, -390. +0.j,   78. +0.j,  104. +0.j,  130. +0.j,
  156. +0.j,   26. +0.j,   52. +0.j,  104. +0.j,   78. +0.j,  156. +0.j,
  195. +0.j,  234. +0.j,   39. +0.j,   78. +0.j,  156. +0.j,  117. +0.j,
  260. +0.j,  312. +0.j,   52. +0.j,  104. +0.j,  208. 

In [15]:
theta = 0.1
evol_gate = PauliEvolutionGate(op, time=theta)
qc = QuantumCircuit(op.num_qubits)
qc.append(evol_gate, range(op.num_qubits))

# Fully decompose and count
qc_opt = transpile(
    qc,
    optimization_level=3,   # 0-3, 3 = heaviest optimization
    basis_gates=['u3','cx'],  # or your hardware basis
    routing_method='sabre'
)

print(qc_opt.count_ops())

OrderedDict([('cx', 72), ('u3', 45)])
